# 🏭 Project 13: Visual Quality Control (QC)
## Notebook 1: Khám Phá & Tiền Xử Lý Dữ Liệu (EDA & Preprocessing)

> **Mục tiêu của notebook:**
> 1. Tải và giải nén bộ dữ liệu **Casting Product Defect** từ Kaggle.
> 2. Kiểm tra số lượng mẫu, tỷ lệ phân phối giữa 2 nhãn (`ok` vs `def`).
> 3. Phân tích thuộc tính ảnh (kích thước, số kênh màu, độ phân giải) và trực quan hóa các mẫu khuyết tật đúc kim loại.
> 4. Chia tập dữ liệu chuẩn: **70% Train - 15% Validation - 15% Test** (Stratified Split).
> 5. Xây dựng Data Augmentation Pipeline chuẩn chỉnh (chỉ áp dụng trên tập Train).

### 1. Cấu hình Môi trường & Kiểm tra GPU

In [ ]:
# Kiểm tra thông số GPU trên Colab
!nvidia-smi

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q kaggle

import os
import glob
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Thiết lập seed để đảm bảo khả năng tái lập (Reproducibility)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

### 2. Tải Dataset Casting Defect từ Kaggle API

> *Lưu ý:* Hãy upload file `kaggle.json` lấy từ tài khoản Kaggle của bạn khi chạy cell dưới đây.

In [ ]:
# Nạp file kaggle.json từ máy tính lên Colab
from google.colab import files
print("Chọn file kaggle.json từ máy của bạn:")
uploaded = files.upload()

# Cấu hình token vào thư mục ~/.kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Tải bộ dữ liệu Casting Product (~830MB)
!kaggle datasets download -d ravirajsinh45/real-life-industrial-dataset-of-casting-product

# Giải nén vào thư mục data_raw
!unzip -q real-life-industrial-dataset-of-casting-product.zip -d data_raw
print("Hoàn tất tải và giải nén dữ liệu!")

### 3. Khám Phá Dữ Liệu (Exploratory Data Analysis - EDA)

In [ ]:
# Kiểm tra cây thư mục giải nén
base_path = "data_raw/casting_data/casting_data"
if not os.path.exists(base_path):
    # Tìm đường dẫn thực tế nếu thư mục lồng nhau khác đi
    for root, dirs, files_list in os.walk("data_raw"):
        if "def_front" in dirs and "ok_front" in dirs:
            base_path = root
            break

print("Đường dẫn thư mục ảnh:", base_path)

# Thu thập toàn bộ đường dẫn ảnh
def_images = glob.glob(os.path.join(base_path, "**/*def_front*/**/*.jpeg"), recursive=True) + \
             glob.glob(os.path.join(base_path, "**/*def_front*/**/*.jpg"), recursive=True)
ok_images = glob.glob(os.path.join(base_path, "**/*ok_front*/**/*.jpeg"), recursive=True) + \
            glob.glob(os.path.join(base_path, "**/*ok_front*/**/*.jpg"), recursive=True)

print(f"Tổng số ảnh khuyết tật (Defective): {len(def_images)}")
print(f"Tổng số ảnh đạt tiêu chuẩn (OK): {len(ok_images)}")
print(f"Tổng cộng: {len(def_images) + len(ok_images)} ảnh")

In [ ]:
# Trực quan hóa tỷ lệ phân phối nhãn (Class Distribution)
labels = ['Sản phẩm Lỗi (Defective)', 'Sản phẩm Đạt (OK)']
counts = [len(def_images), len(ok_images)]
colors = ['#FF4C4C', '#2ECC71']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ cột
sns.barplot(x=labels, y=counts, palette=colors, ax=axes[0])
axes[0].set_title("Số lượng mẫu theo từng nhãn", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Số lượng ảnh")
for i, v in enumerate(counts):
    axes[0].text(i, v + 30, f"{v} ({v/sum(counts)*100:.1f}%)", ha='center', fontweight='bold')

# Biểu đồ tròn
axes[1].pie(counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=140, explode=(0.05, 0))
axes[1].set_title("Tỷ lệ phân phối các lớp", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 4. Kiểm Tra Thuộc Tính & Trực Quan Hóa Mẫu Ảnh

In [ ]:
# Kiểm tra kích thước và số kênh màu ngẫu nhiên
sample_img = cv2.imread(def_images[0])
print(f"Kích thước ảnh mẫu (H, W, C): {sample_img.shape}")
print(f"Kiểu dữ liệu: {sample_img.dtype}")
print(f"Dải giá trị pixel: Min={sample_img.min()}, Max={sample_img.max()}")

In [ ]:
# Vẽ lưới so sánh 5 ảnh Đạt (OK) và 5 ảnh Lỗi (Defect)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

random_def = random.sample(def_images, 5)
random_ok = random.sample(ok_images, 5)

for i, path in enumerate(random_def):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img_rgb)
    axes[0, i].set_title("LỖI (Defective)", color='red', fontweight='bold')
    axes[0, i].axis('off')

for i, path in enumerate(random_ok):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img_rgb)
    axes[1, i].set_title("ĐẠT (OK)", color='green', fontweight='bold')
    axes[1, i].axis('off')

plt.suptitle("So Sánh Mẫu Sản Phẩm Đúc Kim Loại: Lỗi vs Đạt Chuẩn", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 5. Chia Tập Dữ Liệu: Train (70%) / Val (15%) / Test (15%)

> **Quy tắc quan trọng:** Thực hiện Stratified Split để giữ nguyên tỷ lệ nhãn ở cả 3 tập.

In [ ]:
# Tạo DataFrame tổng hợp toàn bộ ảnh và nhãn
data_records = []
for path in def_images:
    data_records.append({'filepath': path, 'label': 1, 'class_name': 'defect'})
for path in ok_images:
    data_records.append({'filepath': path, 'label': 0, 'class_name': 'ok'})

df = pd.DataFrame(data_records)

# Bước 1: Tách Train (70%) và Temp (30%)
train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df['label']
)

# Bước 2: Tách Temp thành Val (15%) và Test (15%)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label']
)

print(f"Tập Train: {len(train_df)} mẫu ({len(train_df)/len(df)*100:.1f}%)")
print(f"Tập Val  : {len(val_df)} mẫu ({len(val_df)/len(df)*100:.1f}%)")
print(f"Tập Test : {len(test_df)} mẫu ({len(test_df)/len(df)*100:.1f}%)")

### 6. Xây Dựng Pipeline Tiền Xử Lý & Data Augmentation

> ⚠️ **Cảnh báo Data Leakage:**  
> - Chỉ áp dụng Data Augmentation trên tập **Train**.
> - Tập **Val** và **Test** chỉ Resize và Rescale pixel về `[0, 1]`.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Pipeline Augmentation dành riêng cho chi tiết cơ khí đúc
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.08),             # Xoay nhẹ +-15 độ
    layers.RandomFlip("horizontal_and_vertical"), # Lật ngang và dọc
    layers.RandomContrast(0.1),              # Biến đổi tương phản nhẹ
    layers.RandomZoom((-0.08, 0.08)),        # Zoom nhẹ
], name="data_augmentation")

# Minh họa hiệu ứng của Data Augmentation trên 1 ảnh mẫu
sample_path = def_images[0]
img = tf.keras.utils.load_img(sample_path, target_size=IMG_SIZE)
img_array = tf.keras.utils.img_to_array(img)
img_batch = tf.expand_dims(img_array, 0)

plt.figure(figsize=(12, 6))
plt.subplot(1, 4, 1)
plt.imshow(img_array.astype("uint8"))
plt.title("Ảnh gốc", fontweight='bold')
plt.axis("off")

for i in range(3):
    augmented_img = data_augmentation(img_batch)
    plt.subplot(1, 4, i + 2)
    plt.imshow(augmented_img[0].numpy().astype("uint8"))
    plt.title(f"Augmented #{i+1}")
    plt.axis("off")

plt.suptitle("Minh Họa Data Augmentation (Chỉ áp dụng trên tập Train)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 7. Xuất Dữ Liệu Chuẩn Bị Cho Huấn Luyện (Notebook 2 & 3)

In [ ]:
# Lưu danh sách train, val, test vào file csv để các notebook tiếp theo sử dụng đồng nhất
os.makedirs("processed_data", exist_ok=True)
train_df.to_csv("processed_data/train.csv", index=False)
val_df.to_csv("processed_data/val.csv", index=False)
test_df.to_csv("processed_data/test.csv", index=False)

print("Đã lưu các file phân chia dữ liệu thành công:")
print(" - processed_data/train.csv")
print(" - processed_data/val.csv")
print(" - processed_data/test.csv")
print("\nSẵn sàng bước sang Notebook 2: 2_Baseline_CNN.ipynb!")